In [5]:
import pandas as pd
import numpy as np

# Configuración
n = 1_000_000
np.random.seed(42)

# Crear DataFrame base con variables del examen
df = pd.DataFrame({
    'client_id': [f'CLI-{i}' for i in range(n)],
    'region': np.random.choice(['North', 'South', 'Center', 'NORTH', 'north '], n, p=[0.4, 0.3, 0.2, 0.05, 0.05]),
    'shipping_service': np.random.choice(['Express', 'Standard', 'Economy', 'EXP', 'std'], n),
    'avg_delivery_time': np.random.normal(5, 2, n),
    'reliability_score': np.random.randint(500, 1000, n),
    'num_shipments': np.random.randint(1, 100, n),
    'avg_parcel_weight': np.random.normal(15, 5, n),
    'total_ship_vol': np.random.randint(100, 5000, n),
    'fuel_consumption': np.random.normal(100, 20, n),
    'total_deliv_cost': np.random.uniform(1000, 10000, n),
    'client_retention_flag': np.random.choice([0, 1], n, p=[0.8, 0.2])
})

# 1. Inyectar Inconsistencias Categóricas (Normalización requerida)
# (Ya integradas en la selección de 'region' y 'shipping_service' arriba)

# 2. Inyectar Valores Imposibles (Negativos donde debe haber positivo)
# Forzamos negativos en avg_delivery_time y fuel_consumption
df.loc[df.sample(frac=0.005).index, 'avg_delivery_time'] = -np.random.uniform(1, 10, int(n*0.005))
df.loc[df.sample(frac=0.003).index, 'fuel_consumption'] = -np.random.uniform(10, 50, int(n*0.003))

# 3. Inyectar Duplicados
# Duplicamos aleatoriamente 1000 filas
dupes = df.sample(1000)
df = pd.concat([df, dupes])

# 4. Inyectar Nulos (En variables clave)
df.loc[df.sample(frac=0.002).index, 'reliability_score'] = np.nan
df.loc[df.sample(frac=0.001).index, 'avg_parcel_weight'] = np.nan

# 5. Valores fuera de lógica extrema (No outliers estadísticos, sino errores físicos)
# Ej: num_shipments no puede ser 99999
df.loc[df.sample(frac=0.001).index, 'num_shipments'] = 99999 

# Sentencia obligatoria final
df.to_csv('logifast_data.csv', index=False)

In [6]:
df.head()

,client_id,region,shipping_service,avg_delivery_time,reliability_score,num_shipments,avg_parcel_weight,total_ship_vol,fuel_consumption,total_deliv_cost,client_retention_flag
0,CLI-0,North,Economy,4.997423,933.0,83,12.566246,4355,124.279830,5388.204758,0
1,CLI-1,north,std,3.562991,598.0,75,10.567509,940,97.854046,3465.771596,1
2,CLI-2,Center,Economy,4.079389,656.0,96,11.507729,183,66.043352,2006.967009,1
3,CLI-3,South,EXP,6.509233,862.0,23,10.329851,328,97.737047,7212.313558,0
4,CLI-4,North,EXP,6.442562,605.0,98,15.952674,943,88.038254,9817.118551,0


In [7]:
def data_health_check(df):
    print("--- 1. Missing Values Report ---")
    missing = df.isnull().sum()
    print(missing[missing > 0])
    
    print("\n--- 2. Duplicated Records ---")
    print(f"Total duplicates: {df.duplicated().sum()}")
    
    print("\n--- 3. Categorical Inconsistencies (Unique Values) ---")
    # Inspección de categorías para detectar variantes como 'NORTH' vs 'North'
    for col in ['region', 'shipping_service']:
        print(f"Unique values in {col}: {df[col].unique()}")

    print("\n--- 4. Logical Constraints Report ---")
    # Detectar valores físicamente imposibles
    neg_delivery = df[df['avg_delivery_time'] < 0].shape[0]
    neg_fuel = df[df['fuel_consumption'] < 0].shape[0]
    impossible_shipments = df[df['num_shipments'] == 99999].shape[0]
    
    print(f"Negative delivery times: {neg_delivery}")
    print(f"Negative fuel consumption: {neg_fuel}")
    print(f"Impossible shipment volume (error 99999): {impossible_shipments}")

# Ejecutar auditoría
data_health_check(df)

# Sentencia obligatoria final
df.to_csv('logifast_data.csv', index=False)

--- 1. Missing Values Report ---
reliability_score    2004
avg_parcel_weight    1001
dtype: int64

--- 2. Duplicated Records ---
Total duplicates: 1000

--- 3. Categorical Inconsistencies (Unique Values) ---
Unique values in region: ['North' 'north ' 'Center' 'South' 'NORTH']
Unique values in shipping_service: ['Economy' 'std' 'EXP' 'Standard' 'Express']

--- 4. Logical Constraints Report ---
Negative delivery times: 11270
Negative fuel consumption: 3001
Impossible shipment volume (error 99999): 1002
